In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.naive_bayes import MultinomialNB, GaussianNB, BernoulliNB

In [12]:
df = pd.read_csv('./../patient2025 - patient2025.csv')
df.head()

,HN,FBS,BMI,Diabetes,Chorestorol,age,hypertension,"vegetarian (1= yes, 0=no)",Marriage Status,Exercise (min/week),Living Area,stroke
0,11223,228.69,34.0,yes,201-220,71.0,1,1,Yes,0.0,Bangkok,1
1,8887,202.21,NaN,No,180-200,52.0,1,0,Yes,90.0,Country,1
2,5666,105.92,30.5,yes,180-200,78.0,1,1,Yes,0.0,Country,1
3,460182,171.23,35.0,No,221-260,54.0,1,0,Yes,0.0,Bangkok,1
4,166665,174.12,28.0,No,180-200,79.0,1,0,Yes,90.0,Country,1


In [13]:
df = df.drop_duplicates(subset="HN")
df = df.drop(columns = ['HN', 'Marriage Status', 'Living Area'])
df

,FBS,BMI,Diabetes,Chorestorol,age,hypertension,"vegetarian (1= yes, 0=no)",Exercise (min/week),stroke
0,228.69,34.0,yes,201-220,71.00,1,1,0.0,1
1,202.21,NaN,No,180-200,52.00,1,0,90.0,1
2,105.92,30.5,yes,180-200,78.00,1,1,0.0,1
3,171.23,35.0,No,221-260,54.00,1,0,0.0,1
4,174.12,28.0,No,180-200,79.00,1,0,90.0,1
...,...,...,...,...,...,...,...,...,...
995,90.51,18.9,yes,Unknown,1.40,0,0,120.0,0
996,118.87,16.3,yes,Unknown,0.24,0,0,120.0,0
997,56.42,31.8,yes,180-200,55.00,0,0,0.0,0
998,73.67,21.0,No,Unknown,29.00,0,0,0.0,0


In [14]:
df["Diabetes"] = df["Diabetes"].replace({
    "yes": 1,
    "No": 0,})
df

C:\Users\Nattanont CSKU\AppData\Local\Temp\ipykernel_8696\2699766147.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Diabetes"] = df["Diabetes"].replace({


,FBS,BMI,Diabetes,Chorestorol,age,hypertension,"vegetarian (1= yes, 0=no)",Exercise (min/week),stroke
0,228.69,34.0,1,201-220,71.00,1,1,0.0,1
1,202.21,NaN,0,180-200,52.00,1,0,90.0,1
2,105.92,30.5,1,180-200,78.00,1,1,0.0,1
3,171.23,35.0,0,221-260,54.00,1,0,0.0,1
4,174.12,28.0,0,180-200,79.00,1,0,90.0,1
...,...,...,...,...,...,...,...,...,...
995,90.51,18.9,1,Unknown,1.40,0,0,120.0,0
996,118.87,16.3,1,Unknown,0.24,0,0,120.0,0
997,56.42,31.8,1,180-200,55.00,0,0,0.0,0
998,73.67,21.0,0,Unknown,29.00,0,0,0.0,0


In [15]:
df['BMI_cat'] = pd.cut(
    df['BMI'],
    bins=[0, 18.5, 25, 30, 35, float('inf')],
    labels=['under', 'normal', 'over', 'obese1', 'obese2']
)

df['FBS_cat'] = pd.cut(
    df['FBS'],
    bins=[0, 100, 126, float('inf')],
    labels=['normal', 'prediabetic', 'diabetic'],
)

df['BMI_cat'] = df['BMI_cat'].cat.add_categories('unknown')
df['BMI_cat'] = df['BMI_cat'].fillna('unknown')

df = pd.get_dummies(df, columns=['BMI_cat', 'Chorestorol', 'FBS_cat'])
df

,FBS,BMI,Diabetes,age,hypertension,"vegetarian (1= yes, 0=no)",Exercise (min/week),stroke,BMI_cat_under,BMI_cat_normal,...,BMI_cat_obese1,BMI_cat_obese2,BMI_cat_unknown,Chorestorol_180-200,Chorestorol_201-220,Chorestorol_221-260,Chorestorol_Unknown,FBS_cat_normal,FBS_cat_prediabetic,FBS_cat_diabetic
0,228.69,34.0,1,71.00,1,1,0.0,1,False,False,...,True,False,False,False,True,False,False,False,False,True
1,202.21,NaN,0,52.00,1,0,90.0,1,False,False,...,False,False,True,True,False,False,False,False,False,True
2,105.92,30.5,1,78.00,1,1,0.0,1,False,False,...,True,False,False,True,False,False,False,False,True,False
3,171.23,35.0,0,54.00,1,0,0.0,1,False,False,...,True,False,False,False,False,True,False,False,False,True
4,174.12,28.0,0,79.00,1,0,90.0,1,False,False,...,False,False,False,True,False,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,90.51,18.9,1,1.40,0,0,120.0,0,False,True,...,False,False,False,False,False,False,True,True,False,False
996,118.87,16.3,1,0.24,0,0,120.0,0,True,False,...,False,False,False,False,False,False,True,False,True,False
997,56.42,31.8,1,55.00,0,0,0.0,0,False,False,...,True,False,False,True,False,False,False,True,False,False
998,73.67,21.0,0,29.00,0,0,0.0,0,False,True,...,False,False,False,False,False,False,True,True,False,False


In [16]:
features = [
    'FBS_cat_normal', 'FBS_cat_prediabetic', 'FBS_cat_diabetic',
    'BMI_cat_under', 'BMI_cat_normal', 'BMI_cat_over', 'BMI_cat_obese1', 'BMI_cat_obese2', 'BMI_cat_unknown', 
    'Chorestorol_180-200', 'Chorestorol_201-220', 'Chorestorol_221-260', 'Chorestorol_Unknown', 
    'Diabetes', 'hypertension', 'vegetarian (1= yes, 0=no)']
X = df[features] # features
y = df.stroke # class

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 6)
model = MultinomialNB()
model.fit(X_train, y_train)
predict_test = model.predict(X_test)

print(f"Accuracy: {metrics.accuracy_score(y_test, predict_test)}")
print(metrics.classification_report(y_test, predict_test))

Accuracy: 0.7666666666666667
              precision    recall  f1-score   support

           0       0.79      0.92      0.85       222
           1       0.60      0.32      0.42        78

    accuracy                           0.77       300
   macro avg       0.69      0.62      0.64       300
weighted avg       0.74      0.77      0.74       300

